# Retrieval Project Submission Notebook

Notebook for final submission generation. Main pipeline stages are visible here, while implementation stays in `src/`.


## Pipeline Overview

This notebook keeps the **main submission flow visible** while delegating implementation to `src/`.

Pipeline stages:
1. detect the runtime and locate the project
2. load and preprocess documents and queries
3. prepare the configured first-stage retriever
4. optionally predict categories for queries
5. optionally build the cross-encoder reranker
6. run first-stage retrieval on the test queries
7. optionally rerank the top candidates
8. write the final Kaggle submission file


In [ ]:
import sys
from pathlib import Path

def detect_runtime_environment() -> str:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return 'colab'
    except Exception:
        if Path('/kaggle/input').exists():
            return 'kaggle'
        return 'local'

def add_project_root_to_syspath(project_name: str = 'retrieval_project') -> None:
    runtime_env = detect_runtime_environment()
    candidates = [Path.cwd(), *Path.cwd().parents]
    if runtime_env == 'colab':
        drive_root = Path('/content/drive/MyDrive')
        if not drive_root.exists():
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
        candidates = [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives'), *candidates]
    elif runtime_env == 'kaggle':
        candidates = [Path('/kaggle/working'), *candidates]

    seen = set()
    for base in candidates:
        key = str(base)
        if key in seen:
            continue
        seen.add(key)
        if (base / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base))
            return
        if (base / project_name / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base / project_name))
            return
        if runtime_env == 'colab' and base.exists():
            for match in base.rglob(project_name):
                if (match / 'src' / 'infra' / 'notebook.py').exists():
                    sys.path.insert(0, str(match))
                    return
    raise FileNotFoundError('Could not locate project root containing src/infra/notebook.py')

add_project_root_to_syspath()

import pandas as pd

from src.infra.notebook import setup_notebook
from src.config import DEFAULT_CONFIG
from src.evaluation import load_ground_truth
from src.pipeline import (
    bootstrap,
    build_cross_encoder_reranker,
    load_project_frames,
    predict_categories,
    prepare_retrievers,
    rerank_retrieval_results,
    run_first_stage_retrieval,
    write_submission,
)

runtime_env, project_root = setup_notebook()
print(f'Detected runtime: {runtime_env}')
print(f'Project root    : {project_root}')

paths, config = bootstrap()
frames = load_project_frames(paths, DEFAULT_CONFIG)
ground_truth = load_ground_truth(paths.data_dir / 'qgts_train.json')

print(f'Runtime environment: {paths.runtime_env}')
print(f'Project directory  : {paths.project_dir}')
print(f'Data directory     : {paths.data_dir}')
print(f'Documents          : {len(frames.docs):,}')
print(f'Train queries      : {len(frames.train_queries):,}')
print(f'Test queries       : {len(frames.test_queries):,}')


## Step 1: Retriever Preparation

The first-stage retriever is prepared here. Depending on the config, this can be TF-IDF, BM25, or embedding-based retrieval.
Artifacts are built once and cached under `cache/` so repeated notebook runs do not recompute them unnecessarily.


In [ ]:
prepared_retrievers = prepare_retrievers(frames, paths, config=config)
print(f'Prepared retrievers: {sorted(prepared_retrievers.keys())}')


## Step 2: Category Prediction

If category filtering is enabled, the notebook trains or loads the lightweight category classifier, predicts categories for train and test queries, and builds a document-category lookup.
This category information is later used either to filter retrieval or to apply a soft bonus during reranking.


In [ ]:
category_artifacts = predict_categories(frames, paths, ground_truth=ground_truth, config=config)
if category_artifacts.classifier_artifacts is None:
    print('Category filtering disabled in config.')
else:
    print(f'Train category predictions: {len(category_artifacts.train_query_category_map):,}')
    print(f'Test category predictions : {len(category_artifacts.test_query_category_map):,}')
    print(f'Category accuracy         : {category_artifacts.classifier_accuracy:.5f}')


## Step 3: Cross-Encoder Reranker

If reranking is enabled, the notebook loads or trains the cross-encoder model from the training relevance labels.
This is a second-stage model: it does not replace retrieval, it only reorders the candidate list returned by the first stage.


In [ ]:
cross_encoder_reranker = build_cross_encoder_reranker(frames, paths, ground_truth, config=config)
if cross_encoder_reranker is None:
    print('Cross-encoder reranking disabled in config.')
else:
    print('Cross-encoder reranker is ready.')


## Step 4: First-Stage Retrieval

This cell runs the retrieval stage on the **test queries**.
If category filtering is enabled, search is restricted to the predicted category; otherwise the configured final retriever searches the full corpus.


In [ ]:
test_results, test_category_predictions = run_first_stage_retrieval(
    frames=frames,
    paths=paths,
    prepared_retrievers=prepared_retrievers,
    category_artifacts=category_artifacts,
    split='test',
    config=config,
)
print(f'First-stage results: {len(test_results):,} queries')


## Step 5: Optional Reranking

When available, the cross-encoder rescoring step is applied to the top retrieved candidates.
The reranker can also add a category bonus when the predicted query category matches the document category.


In [ ]:
test_results = rerank_retrieval_results(
    results=test_results,
    frames=frames,
    category_artifacts=category_artifacts,
    cross_encoder_reranker=cross_encoder_reranker,
    split='test',
    config=config,
)
print('Reranking step completed.')


## Step 6: Submission Writing

The final ranked results are serialized into the exact Kaggle submission format using the sample submission schema.
The preview below is only for inspection; the important output is the CSV written to `paths.output_path`.


In [ ]:
write_submission(test_results, paths, category_predictions=test_category_predictions)
print(f'Submission written to: {paths.output_path}')
submission_preview = pd.read_csv(paths.output_path)
submission_preview.head()
